In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Caminhos do Bronze
BRONZE = "/Volumes/workspace/olist/bronze"
SILVER = "/Volumes/workspace/olist/silver"

# Lê as 4 tabelas do Bronze
df_orders      = spark.read.format("delta").load(f"{BRONZE}/orders")
df_order_items = spark.read.format("delta").load(f"{BRONZE}/orders_items")
df_customers   = spark.read.format("delta").load(f"{BRONZE}/customers")
df_products    = spark.read.format("delta").load(f"{BRONZE}/products")

# Confirma leitura
print(f"orders:      {df_orders.count()} registros")
print(f"order_items: {df_order_items.count()} registros")
print(f"customers:   {df_customers.count()} registros")
print(f"products:    {df_products.count()} registros")

In [0]:
# ORDERS - remove colunas de auditoria, padroniza status
df_orders_silver = df_orders \
    .drop("_ingest_timestamp", "_source_filename", "_processing_date", "_execucao_id") \
    .filter(F.col("order_id").isNotNull()) \
    .filter(F.col("customer_id").isNotNull()) \
    .withColumn("order_status", F.lower(F.trim(F.col("order_status"))))

# ORDER_ITEMS - cast de valores monetários
df_order_items_silver = df_order_items \
    .drop("_ingest_timestamp", "_source_filename", "_processing_date", "_execucao_id") \
    .filter(F.col("order_id").isNotNull()) \
    .withColumn("price", F.col("price").cast("double")) \
    .withColumn("freight_value", F.col("freight_value").cast("double"))

#CUSTOMERS -padroniza cidade e estado
df_customers_silver = df_customers \
    .drop("_ingest_timestamp", "_source_filename", "_processing_date", "_execucao_id") \
    .filter(F.col("customer_id").isNotNull()) \
    .withColumn("customer_city", F.lower(F.trim(F.col("customer_city")))) \
    .withColumn("customer_state", F.lower(F.trim(F.col("customer_state"))))

# PRODUCTS - remove colunas de auditoria
df_products_silver = df_products \
    .drop("ingest_timestamp", "source_filename", "processing_date", "execucao_id") \
    .filter(F.col("product_id").isNotNull()) \
    .withColumn("product_category_name", F.coalesce(F.col("product_category_name"), F.lit("unknown")))

print("Limpeza concluída!")
print(f"orders_silver:      {df_orders_silver.count()} registros")
print(f"order_items_silver: {df_order_items_silver.count()} registros")
print(f"customers_silver:   {df_customers_silver.count()} registros")
print(f"products_silver:    {df_products_silver.count()} registros")

In [0]:
# Join principal - uma linha por item de pedido
df_silver = df_order_items_silver \
    .join(df_orders_silver, on="order_id", how="left") \
    .join(df_customers_silver, on="customer_id", how="left") \
    .join(df_products_silver, on="product_id", how="left")

print(f"Total de registros: {df_silver.count()}")
print(f"Total de colunas:   {len(df_silver.columns)}")
print(f"\nColunas: {df_silver.columns}")

In [0]:
# Remove colunas de auditoria e corrige typos
df_silver = df_silver \
    .drop("_ingest_timestamp", "_source_filename", "_processing_date", "_execucao_id") \
    .withColumnRenamed("product_name_lenght",        "product_name_length") \
    .withColumnRenamed("product_description_lenght", "product_description_length")

print(f"Total de registros: {df_silver.count()}")
print(f"Total de colunas:   {len(df_silver.columns)}")
print(f"\nColunas: {df_silver.columns}")

In [0]:
from pyspark.sql import Window

# Janelas de cálculo
window_pedido = Window.partitionBy("order_id")
window_cliente = Window.partitionBy("customer_id") \
                        .orderBy("order_purchase_timestamp") \
                        .rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_silver = df_silver \
    .withColumn("valor_total_pedido",
        F.sum(F.col("price") + F.col("freight_value")).over(window_pedido)) \
    .withColumn("rank_item_no_pedido",
        F.row_number().over(Window.partitionBy("order_id").orderBy(F.col("price").desc()))) \
    .withColumn("ltv_acumulado",
                 F.sum(F.col("price") + F.col("freight_value")).over(window_cliente))
    
print(f" Colunas após window functions: {len(df_silver.columns)}")
display(df_silver.select(
    "order_id",
    "customer_id", 
    "price",
    "freight_value",
    "valor_total_pedido",
    "rank_item_no_pedido",
    "ltv_acumulado"
).limit(10))

In [0]:
from datetime import date as dt

df_silver \
    .withColumn("_processing_date", F.lit(str(dt.today()))) \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("order_status", "_processing_date") \
    .save(f"{SILVER}/orders_enriched")

total = spark.read.format("delta").load(f"{SILVER}/orders_enriched").count()
print(f"Silver gravado: {total} registros")